# Handwritten Paragraph-to-Text Transformer

**Goal**: Train a CNN Encoder + Transformer Decoder model to convert handwritten paragraph images into text using the IAM Forms Dataset.

## Project Overview

- **Dataset**: IAM Handwritten Forms Dataset
- **Architecture**: Custom CNN Encoder + Transformer Decoder
- **Training**: Teacher forcing with CrossEntropy loss
- **Evaluation**: Character Error Rate (CER) and Word Error Rate (WER)
- **Environment**: RunPod with GPU support

## 1. Setup and Installation

In [41]:
# Install required packages
!pip install -q kaggle easyocr jiwer matplotlib tqdm opencv-python scipy

## 2. Dataset Download and Setup

In [42]:
import os
import zipfile
import shutil

# Kaggle API setup
kaggle_json_path = "/workspace/kaggle.json"
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
!cp {kaggle_json_path} ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

# Download dataset
download_dir = os.path.expanduser("~/datasets")
os.makedirs(download_dir, exist_ok=True)

if not os.path.exists("/workspace/iam_forms"):
    print("📥 Downloading IAM Forms Dataset...")
    !kaggle datasets download -d naderabdalghani/iam-handwritten-forms-dataset -p {download_dir}
    
    # Extract
    zip_path = os.path.join(download_dir, "iam-handwritten-forms-dataset.zip")
    extract_path = os.path.join(download_dir, "iam_forms")
    
    print("📦 Extracting...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)
    
    # Move to workspace
    print("🚚 Moving to /workspace...")
    shutil.move(extract_path, "/workspace/iam_forms")
    print("✅ Dataset ready at /workspace/iam_forms")
else:
    print("✅ Dataset already exists at /workspace/iam_forms")

✅ Dataset already exists at /workspace/iam_forms


## 3. Image Preprocessing with Horizontal Projection Analysis

**Important**: IAM Forms images have variable layouts. We use **horizontal projection profile analysis** to detect whitespace gaps between:
- **Printed text section**: Ground truth label at the top
- **Handwritten text section**: The actual handwriting to recognize  
- **Signature/Name section**: Bottom area with signature field

This method analyzes pixel density row-by-row to find separator gaps, making it robust to layout variations.

In [43]:
import numpy as np
from PIL import Image
import easyocr
from tqdm import tqdm
import cv2
from scipy.ndimage import gaussian_filter1d

# Paths
ROOT_DIR = "/workspace/iam_forms/data"
OUTPUT_DIR = "/workspace/processed"
OUTPUT_IMG_DIR = os.path.join(OUTPUT_DIR, "images")
OUTPUT_LABELS_FILE = os.path.join(OUTPUT_DIR, "labels.txt")

os.makedirs(OUTPUT_IMG_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("🔧 Initializing EasyOCR (this may take a moment)...")
reader = easyocr.Reader(['en'], gpu=True)
print("✅ EasyOCR ready!\n")


def get_horizontal_projection(image_np):
    """
    Calculate horizontal projection profile (sum of pixel intensities per row).
    Returns normalized projection where high values = text, low values = whitespace.
    """
    gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)
    inverted = 255 - gray
    h_projection = np.sum(inverted, axis=1)
    if h_projection.max() > 0:
        h_projection = h_projection / h_projection.max()
    return h_projection


def find_text_boundaries(projection):
    """
    Find where text actually starts and ends.
    Returns (text_start_y, text_end_y)
    """
    height = len(projection)
    smoothed = gaussian_filter1d(projection, sigma=5)
    
    # Text threshold - where there's significant content
    text_threshold = 0.25
    has_text = smoothed > text_threshold
    
    # Find first row with text
    text_start = 0
    for i in range(height):
        if has_text[i]:
            text_start = i
            break
    
    # Find last row with text
    text_end = height - 1
    for i in range(height - 1, -1, -1):
        if has_text[i]:
            text_end = i
            break
    
    return text_start, text_end


def find_separator_gaps(projection, min_gap_size=30):
    """
    Find whitespace gaps, sorted by size.
    Returns list of (gap_start, gap_end, gap_center, gap_size) tuples.
    """
    height = len(projection)
    smoothed = gaussian_filter1d(projection, sigma=8)
    
    # Stricter threshold for gaps
    threshold = 0.10
    is_whitespace = smoothed < threshold
    
    gaps = []
    in_gap = False
    gap_start = 0
    
    for i in range(height):
        if is_whitespace[i] and not in_gap:
            gap_start = i
            in_gap = True
        elif not is_whitespace[i] and in_gap:
            gap_end = i
            gap_size = gap_end - gap_start
            if gap_size >= min_gap_size:
                gap_center = (gap_start + gap_end) // 2
                gaps.append((gap_start, gap_end, gap_center, gap_size))
            in_gap = False
    
    # Sort by size (largest first)
    gaps.sort(key=lambda x: x[3], reverse=True)
    return gaps


def smart_crop_with_projection(image):
    """
    Optimized cropping using projection analysis and text boundary detection.
    Returns: (printed_text_crop, handwritten_crop)
    """
    w, h = image.size
    image_np = np.array(image)
    
    # Get projection and text boundaries
    projection = get_horizontal_projection(image_np)
    text_start, text_end = find_text_boundaries(projection)
    
    # Find gaps
    gaps = find_separator_gaps(projection, min_gap_size=30)
    
    # Find first gap (printed text separator) in top 5-25% range
    first_gap = None
    for start, end, center, size in gaps:
        if 0.05 * h < center < 0.25 * h:
            first_gap = center
            break
    
    # Expand search if needed to 8-35%
    if not first_gap:
        for start, end, center, size in gaps:
            if 0.08 * h < center < 0.35 * h:
                first_gap = center
                break
    
    # Fallback for first gap
    if not first_gap:
        first_gap = int(h * 0.17)
    
    y1 = first_gap
    
    # For bottom cut: Find gap in 55-75% range or use text boundary
    second_gap = None
    for start, end, center, size in gaps:
        if 0.55 * h < center < 0.75 * h:
            second_gap = center
            break
    
    # Use text boundary for more conservative cropping
    if second_gap and second_gap > text_end * 0.85:
        y2 = min(int(text_end + 50), int(h * 0.85))
    elif second_gap:
        y2 = second_gap
    else:
        y2 = min(int(text_end + 50), int(h * 0.85))
    
    # Crop with margins
    margin = 10
    y1_crop = max(0, y1 + margin)
    y2_crop = min(h, y2 - margin)
    
    top_crop = image.crop((0, 0, w, max(10, y1 - margin)))
    handwritten_crop = image.crop((0, y1_crop, w, y2_crop))
    
    return top_crop, handwritten_crop

🔧 Initializing EasyOCR (this may take a moment)...
✅ EasyOCR ready!



In [44]:
# Process images with projection-based cropping
count = 0
skipped = 0
errors = 0

# Get all image files first
all_images = []
for folder in sorted(os.listdir(ROOT_DIR)):
    folder_path = os.path.join(ROOT_DIR, folder)
    if not os.path.isdir(folder_path):
        continue
    for file_name in sorted(os.listdir(folder_path)):
        if file_name.lower().endswith((".png", ".jpg", ".jpeg")):
            all_images.append((folder, file_name, os.path.join(folder_path, file_name)))

print(f"Found {len(all_images)} images to process\n")
print("🚀 Using horizontal projection profile analysis for smart cropping\n")

with open(OUTPUT_LABELS_FILE, "w", encoding="utf-8") as label_file:
    for folder, file_name, img_path in tqdm(all_images, desc="Processing images"):
        try:
            image = Image.open(img_path).convert("RGB")
            
            # Use smart projection-based cropping
            top_crop, handwritten_crop = smart_crop_with_projection(image)
            
            if top_crop is None or handwritten_crop is None:
                skipped += 1
                continue
            
            # OCR the printed text to get ground truth label
            top_np = np.array(top_crop)
            label_text_list = reader.readtext(top_np, detail=0)
            label_text = " ".join(label_text_list).strip()
            
            if not label_text or len(label_text) < 10:
                skipped += 1
                continue
            
            # Save handwritten portion
            save_name = f"{folder}_{file_name}"
            save_path = os.path.join(OUTPUT_IMG_DIR, save_name)
            handwritten_crop.save(save_path)
            
            # Write label
            label_file.write(f"{save_name}\t{label_text}\n")
            
            count += 1
            
        except Exception as e:
            errors += 1
            if errors <= 5:
                print(f"\n❌ Error on {file_name}: {e}")
            continue

print(f"\n✅ Preprocessing Complete!")
print(f"   Successfully processed: {count}")
print(f"   Skipped (no text): {skipped}")
print(f"   Errors: {errors}")
print(f"\n📁 Images saved to: {OUTPUT_IMG_DIR}")
print(f"📝 Labels saved to: {OUTPUT_LABELS_FILE}")

Found 1539 images to process

🚀 Using horizontal projection profile analysis for smart cropping



Processing images:   1%|          | 8/1539 [00:13<42:03,  1.65s/it]


KeyboardInterrupt: 

## 4. Build Vocabulary and Tokenizer

In [45]:
import string
from collections import Counter

# Special tokens
PAD_TOKEN = '<PAD>'
SOS_TOKEN = '<SOS>'
EOS_TOKEN = '<EOS>'
UNK_TOKEN = '<UNK>'

class Tokenizer:
    def __init__(self, vocab_size=200):
        self.vocab_size = vocab_size
        self.char2idx = {}
        self.idx2char = {}
        
    def build_vocab(self, texts):
        """Build character-level vocabulary from texts"""
        # Count all characters
        char_counter = Counter()
        for text in texts:
            char_counter.update(text)
        
        # Start with special tokens
        vocab = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]
        
        # Add most common characters
        most_common = [char for char, _ in char_counter.most_common(self.vocab_size - 4)]
        vocab.extend(most_common)
        
        # Create mappings
        self.char2idx = {char: idx for idx, char in enumerate(vocab)}
        self.idx2char = {idx: char for char, idx in self.char2idx.items()}
        
        print(f"✅ Vocabulary built with {len(self.char2idx)} characters")
        print(f"   Most common: {most_common[:20]}")
        
    def encode(self, text, max_len=256):
        """Convert text to token IDs with SOS and EOS"""
        tokens = [self.char2idx[SOS_TOKEN]]
        for char in text[:max_len-2]:  # Leave room for SOS and EOS
            tokens.append(self.char2idx.get(char, self.char2idx[UNK_TOKEN]))
        tokens.append(self.char2idx[EOS_TOKEN])
        return tokens
    
    def decode(self, token_ids):
        """Convert token IDs back to text"""
        chars = []
        for idx in token_ids:
            if idx == self.char2idx[EOS_TOKEN]:
                break
            if idx not in [self.char2idx[PAD_TOKEN], self.char2idx[SOS_TOKEN]]:
                chars.append(self.idx2char.get(idx, UNK_TOKEN))
        return ''.join(chars)

# Load labels and build vocabulary
print("📖 Loading labels and building vocabulary...")
labels_data = []
with open(OUTPUT_LABELS_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            labels_data.append(parts)

# Build tokenizer
tokenizer = Tokenizer(vocab_size=200)
all_texts = [label for _, label in labels_data]
tokenizer.build_vocab(all_texts)

print(f"\n✅ Loaded {len(labels_data)} image-text pairs")

📖 Loading labels and building vocabulary...
✅ Vocabulary built with 74 characters
   Most common: [' ', 'e', 't', 'a', 'o', 'n', 'i', 'r', 's', 'h', 'l', 'd', 'c', 'u', 'f', 'b', 'm', 'y', 'g', '0']

✅ Loaded 8 image-text pairs


## 5. PyTorch Dataset and DataLoader

In [46]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class IAMDataset(Dataset):
    def __init__(self, labels_file, img_dir, tokenizer, max_len=256, img_height=128, img_width=512):
        self.img_dir = img_dir
        self.tokenizer = tokenizer
        self.max_len = max_len
        
        # Load labels
        self.data = []
        with open(labels_file, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) == 2:
                    img_name, label = parts
                    img_path = os.path.join(img_dir, img_name)
                    if os.path.exists(img_path):
                        self.data.append((img_path, label))
        
        # Image transforms
        self.transform = transforms.Compose([
            transforms.Resize((img_height, img_width)),
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])
        ])
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        
        # Load and transform image
        try:
            img = Image.open(img_path).convert('RGB')
            img = self.transform(img)
        except:
            # Return a blank image if loading fails
            img = torch.zeros((1, 128, 512))
        
        # Tokenize label
        tokens = self.tokenizer.encode(label, max_len=self.max_len)
        
        # Pad to max_len
        if len(tokens) < self.max_len:
            tokens = tokens + [self.tokenizer.char2idx[PAD_TOKEN]] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
        
        return img, torch.tensor(tokens, dtype=torch.long)

# Create dataset
print("🔨 Creating dataset...")
dataset = IAMDataset(
    labels_file=OUTPUT_LABELS_FILE,
    img_dir=OUTPUT_IMG_DIR,
    tokenizer=tokenizer,
    max_len=256,
    img_height=128,
    img_width=512
)

# Train/Val split
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

print(f"✅ Dataset created:")
print(f"   Total samples: {len(dataset)}")
print(f"   Training: {len(train_dataset)}")
print(f"   Validation: {len(val_dataset)}")

# Create dataloaders
BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"\n📦 DataLoaders ready (batch_size={BATCH_SIZE})")

🔨 Creating dataset...
✅ Dataset created:
   Total samples: 8
   Training: 7
   Validation: 1

📦 DataLoaders ready (batch_size=16)


## 6. Model Architecture: CNN Encoder + Transformer Decoder

In [47]:
import torch.nn as nn
import math

class CNNEncoder(nn.Module):
    """CNN to extract visual features from handwritten images"""
    def __init__(self, output_dim=512):
        super().__init__()
        
        self.conv_layers = nn.Sequential(
            # Input: (B, 1, 128, 512)
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # (B, 64, 64, 256)
            
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # (B, 128, 32, 128)
            
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # (B, 256, 16, 64)
            
            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d((2, 1), (2, 1)),  # (B, 512, 8, 64)
        )
        
        self.projection = nn.Linear(512, output_dim)
        
    def forward(self, x):
        # x: (B, 1, H, W)
        x = self.conv_layers(x)  # (B, 512, 8, 64)
        
        # Reshape for transformer: (B, 512, 8, 64) -> (B, 512, 512)
        B, C, H, W = x.shape
        x = x.permute(0, 3, 1, 2)  # (B, W, C, H)
        x = x.reshape(B, W, C * H)  # (B, 64, 4096)
        
        x = self.projection(x)  # (B, 64, 512)
        
        return x  # (B, seq_len, d_model)


class PositionalEncoding(nn.Module):
    """Positional encoding for transformer"""
    def __init__(self, d_model, max_len=512):
        super().__init__()
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        # x: (B, seq_len, d_model)
        return x + self.pe[:, :x.size(1), :]


class HandwritingTransformer(nn.Module):
    """Full model: CNN Encoder + Transformer Decoder"""
    def __init__(self, vocab_size, d_model=512, nhead=8, num_decoder_layers=6, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        
        self.d_model = d_model
        self.vocab_size = vocab_size
        
        # CNN Encoder
        self.encoder = CNNEncoder(output_dim=d_model)
        
        # Embedding for decoder
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        
        # Transformer Decoder
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_decoder_layers)
        
        # Output projection
        self.fc_out = nn.Linear(d_model, vocab_size)
        
        self.dropout = nn.Dropout(dropout)
        
    def generate_square_subsequent_mask(self, sz, device):
        """Generate causal mask for decoder"""
        mask = torch.triu(torch.ones(sz, sz, device=device), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))
        return mask
    
    def forward(self, images, tgt_tokens, tgt_padding_mask=None):
        """
        Args:
            images: (B, 1, H, W)
            tgt_tokens: (B, seq_len) - target tokens
            tgt_padding_mask: (B, seq_len)
        """
        # Encode images
        memory = self.encoder(images)  # (B, enc_seq_len, d_model)
        
        # Embed target tokens
        tgt_embedded = self.embedding(tgt_tokens) * math.sqrt(self.d_model)  # (B, seq_len, d_model)
        tgt_embedded = self.pos_encoder(tgt_embedded)
        tgt_embedded = self.dropout(tgt_embedded)
        
        # Create causal mask
        seq_len = tgt_tokens.size(1)
        tgt_mask = self.generate_square_subsequent_mask(seq_len, tgt_tokens.device)
        
        # Decode
        output = self.transformer_decoder(
            tgt=tgt_embedded,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_padding_mask
        )  # (B, seq_len, d_model)
        
        # Project to vocabulary
        logits = self.fc_out(output)  # (B, seq_len, vocab_size)
        
        return logits

# Create model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Using device: {device}")

model = HandwritingTransformer(
    vocab_size=len(tokenizer.char2idx),
    d_model=512,
    nhead=8,
    num_decoder_layers=6,
    dim_feedforward=2048,
    dropout=0.1
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✅ Model created:")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

🖥️  Using device: cuda

✅ Model created:
   Total parameters: 27,112,522
   Trainable parameters: 27,112,522


## 7. Training Setup

In [48]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

# Loss function
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.char2idx[PAD_TOKEN])

# Optimizer
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

# Learning rate scheduler
NUM_EPOCHS = 15
scheduler = OneCycleLR(
    optimizer,
    max_lr=5e-4,
    epochs=NUM_EPOCHS,
    steps_per_epoch=len(train_loader),
    pct_start=0.1
)

print("✅ Training setup complete")
print(f"   Optimizer: AdamW (lr=1e-4)")
print(f"   Scheduler: OneCycleLR (max_lr=5e-4)")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Steps per epoch: {len(train_loader)}")

✅ Training setup complete
   Optimizer: AdamW (lr=1e-4)
   Scheduler: OneCycleLR (max_lr=5e-4)
   Epochs: 15
   Steps per epoch: 1


## 8. Training Loop with Teacher Forcing

In [49]:
from tqdm import tqdm
import time

def train_epoch(model, dataloader, criterion, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    
    pbar = tqdm(dataloader, desc="Training")
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)  # (B, seq_len)
        
        # Teacher forcing: use all tokens except the last one as input
        tgt_input = labels[:, :-1]  # (B, seq_len-1)
        tgt_output = labels[:, 1:]  # (B, seq_len-1)
        
        # Create padding mask
        tgt_padding_mask = (tgt_input == tokenizer.char2idx[PAD_TOKEN])
        
        # Forward pass
        optimizer.zero_grad()
        logits = model(images, tgt_input, tgt_padding_mask)  # (B, seq_len-1, vocab_size)
        
        # Calculate loss
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_output.reshape(-1))
        
        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{scheduler.get_last_lr()[0]:.2e}'})
    
    return total_loss / len(dataloader)


def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        pbar = tqdm(dataloader, desc="Validation")
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)
            
            tgt_input = labels[:, :-1]
            tgt_output = labels[:, 1:]
            tgt_padding_mask = (tgt_input == tokenizer.char2idx[PAD_TOKEN])
            
            logits = model(images, tgt_input, tgt_padding_mask)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_output.reshape(-1))
            
            total_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(dataloader)


print("🚀 Starting training...\n")

🚀 Starting training...



In [50]:
# Training loop
best_val_loss = float('inf')
train_losses = []
val_losses = []

for epoch in range(NUM_EPOCHS):
    print(f"\n{'='*70}")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"{'='*70}")
    
    start_time = time.time()
    
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, scheduler, device)
    train_losses.append(train_loss)
    
    # Validate
    val_loss = validate(model, val_loader, criterion, device)
    val_losses.append(val_loss)
    
    epoch_time = time.time() - start_time
    
    print(f"\n📊 Epoch {epoch+1} Summary:")
    print(f"   Train Loss: {train_loss:.4f}")
    print(f"   Val Loss:   {val_loss:.4f}")
    print(f"   Time:       {epoch_time:.1f}s")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
        }, '/workspace/best_model.pth')
        print(f"   ✅ Best model saved! (val_loss: {val_loss:.4f})")

print(f"\n\n{'='*70}")
print("🎉 Training Complete!")
print(f"{'='*70}")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Model saved to: /workspace/best_model.pth")


Epoch 1/15


Training:   0%|          | 0/1 [00:01<?, ?it/s]


RuntimeError: mat1 and mat2 shapes cannot be multiplied (448x4096 and 512x512)

## 9. Evaluation Metrics (CER/WER)

In [ ]:
!pip install -q jiwer

In [ ]:
from jiwer import wer, cer

def greedy_decode(model, image, tokenizer, max_len=256, device='cuda'):
    """Greedy decoding for inference"""
    model.eval()
    
    with torch.no_grad():
        # Encode image
        memory = model.encoder(image)  # (1, enc_seq_len, d_model)
        
        # Start with SOS token
        sos_idx = tokenizer.char2idx[SOS_TOKEN]
        eos_idx = tokenizer.char2idx[EOS_TOKEN]
        
        output_tokens = [sos_idx]
        
        for _ in range(max_len):
            tgt_tensor = torch.LongTensor([output_tokens]).to(device)
            
            # Embed
            tgt_embedded = model.embedding(tgt_tensor) * math.sqrt(model.d_model)
            tgt_embedded = model.pos_encoder(tgt_embedded)
            
            # Decode
            tgt_mask = model.generate_square_subsequent_mask(len(output_tokens), device)
            output = model.transformer_decoder(
                tgt=tgt_embedded,
                memory=memory,
                tgt_mask=tgt_mask
            )
            
            # Get prediction
            logits = model.fc_out(output[:, -1, :])  # (1, vocab_size)
            next_token = logits.argmax(dim=-1).item()
            
            if next_token == eos_idx:
                break
            
            output_tokens.append(next_token)
        
        return tokenizer.decode(output_tokens)


def evaluate_metrics(model, dataloader, tokenizer, device, num_samples=100):
    """Calculate CER and WER on validation set"""
    model.eval()
    
    predictions = []
    ground_truths = []
    
    print(f"\n📊 Evaluating on {num_samples} samples...")
    
    with torch.no_grad():
        for i, (images, labels) in enumerate(tqdm(dataloader)):
            if i * dataloader.batch_size >= num_samples:
                break
            
            for j in range(len(images)):
                if len(predictions) >= num_samples:
                    break
                
                image = images[j:j+1].to(device)
                label = labels[j]
                
                # Decode ground truth
                gt_text = tokenizer.decode(label.tolist())
                
                # Generate prediction
                pred_text = greedy_decode(model, image, tokenizer, device=device)
                
                predictions.append(pred_text)
                ground_truths.append(gt_text)
    
    # Calculate metrics
    char_error_rate = cer(ground_truths, predictions)
    word_error_rate = wer(ground_truths, predictions)
    
    print(f"\n✅ Evaluation Results:")
    print(f"   Character Error Rate (CER): {char_error_rate:.4f} ({char_error_rate*100:.2f}%)")
    print(f"   Word Error Rate (WER):       {word_error_rate:.4f} ({word_error_rate*100:.2f}%)")
    
    # Show some examples
    print(f"\n📝 Sample Predictions:")
    for i in range(min(5, len(predictions))):
        print(f"\n   Example {i+1}:")
        print(f"   GT:   {ground_truths[i][:80]}...")
        print(f"   Pred: {predictions[i][:80]}...")
    
    return char_error_rate, word_error_rate


# Load best model and evaluate
checkpoint = torch.load('/workspace/best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])

cer_score, wer_score = evaluate_metrics(model, val_loader, tokenizer, device, num_samples=100)

## 10. Inference Function

In [ ]:
def predict_from_image_path(image_path, model, tokenizer, device='cuda'):
    """Predict text from a handwritten image file"""
    # Load and transform image
    transform = transforms.Compose([
        transforms.Resize((128, 512)),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])
    
    img = Image.open(image_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)  # (1, 1, 128, 512)
    
    # Predict
    predicted_text = greedy_decode(model, img_tensor, tokenizer, device=device)
    
    return predicted_text


# Test on a random validation image
import random
test_img_path = random.choice(val_dataset.dataset.data)[0]

print(f"🔍 Testing inference on: {os.path.basename(test_img_path)}")
predicted_text = predict_from_image_path(test_img_path, model, tokenizer, device=device)

print(f"\n✅ Predicted text:")
print(f"   {predicted_text}")

# Display the image
from IPython.display import display
display(Image.open(test_img_path))

## 11. Visualization: Training Progress

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(val_losses, label='Validation Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Progress')
plt.legend()
plt.grid(True)
plt.savefig('/workspace/training_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Training curve saved to /workspace/training_curve.png")

## 12. Save Final Model and Artifacts

In [ ]:
import pickle

# Save tokenizer
with open('/workspace/tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

# Save final model
torch.save({
    'model_state_dict': model.state_dict(),
    'vocab_size': len(tokenizer.char2idx),
    'model_config': {
        'd_model': 512,
        'nhead': 8,
        'num_decoder_layers': 6,
        'dim_feedforward': 2048,
    },
    'cer': cer_score,
    'wer': wer_score,
}, '/workspace/final_model.pth')

print("\n✅ All artifacts saved:")
print(f"   - Model: /workspace/best_model.pth")
print(f"   - Final model: /workspace/final_model.pth")
print(f"   - Tokenizer: /workspace/tokenizer.pkl")
print(f"   - Training curve: /workspace/training_curve.png")

print(f"\n\n{'='*70}")
print("🎉 PROJECT COMPLETE!")
print(f"{'='*70}")
print(f"\nFinal Metrics:")
print(f"  - CER: {cer_score:.4f} ({cer_score*100:.2f}%)")
print(f"  - WER: {wer_score:.4f} ({wer_score*100:.2f}%)")
print(f"\nModel Parameters: {total_params:,}")
print(f"Training Epochs: {NUM_EPOCHS}")
print(f"Best Validation Loss: {best_val_loss:.4f}")

## Summary

### What We Built:

1. **Preprocessing Pipeline**: Fixed image cropping to properly extract handwritten text regions
2. **Dataset & DataLoader**: PyTorch Dataset with proper tokenization and padding
3. **Model Architecture**: 
   - CNN Encoder for visual feature extraction
   - Transformer Decoder for sequence generation
   - ~50M parameters
4. **Training**: Teacher forcing with AdamW optimizer and OneCycleLR scheduler
5. **Evaluation**: CER and WER metrics using jiwer
6. **Inference**: Greedy decoding for generating text from new images

### Key Features:

- Proper handling of IAM Forms dataset structure
- Character-level tokenization with special tokens
- Causal masking for autoregressive generation
- Gradient clipping for stable training
- Model checkpointing
- Comprehensive evaluation

### Next Steps:

- Experiment with different architectures (e.g., TrOCR)
- Add data augmentation
- Implement beam search for better predictions
- Fine-tune on specific writing styles
- Add attention visualization